# Treino g1_v3 no Kaggle

Mesmo treino do `treino_runpod.ipynb`, pelo mesmo `entrada.sh`. Muda só o que o Kaggle exige.

- **Settings → Accelerator:** GPU T4 x2. O treino usa uma das duas. A P100 não serve: o torch do Kaggle não roda nela.
- **Settings → Internet:** On (clone do repo e `pip`). Exige conta com telefone verificado.
- **Retomada:** suba o checkpoint como Dataset do Kaggle e anexe ao notebook em **Add Input**. A célula 4 acha o arquivo em `/kaggle/input`, confere o md5 e monta a pasta da corrida.
- **Sem vídeo e sem TensorBoard:** o acompanhamento é o log, na última célula.
- **Para guardar os checkpoints:** use **Save Version → Save & Run All**. A saída de `/kaggle/working` fica na versão quando a sessão termina. Numa sessão interativa, baixe os arquivos antes de fechar.
- `TIMEOUT_H` fica abaixo do limite da sessão do Kaggle: o treino para sozinho e a última célula termina, o que deixa a versão gravar a saída.

In [ ]:
# Configuração. Única célula que você precisa editar.
import re

RUN = "unitree2"        # nome da corrida: letras, números, '.', '-' e '_'
RETOMA = 1              # 0 = corrida nova; 1 = retoma do checkpoint CKPT
ITERS = 30000           # iterações desta sessão (na retomada, somam ao início); o TIMEOUT_H corta antes
NUM_ENVS = 4096
TIMEOUT_H = 8           # teto de horas do treino; a sessão do Kaggle tem limite próprio (9 h ou 12 h, conforme a fonte)
CKPT = "model_10850.pt"                          # nome do checkpoint dentro de /kaggle/input (aceita sufixo, ex.: .ptrom)
CKPT_MD5 = "6b795bd6b9ffb1ed3704049ec4089cac"    # md5 do arquivo local; "" pula a conferência
RAIZ = "/kaggle/working/g1_training"
SAIDA = "/kaggle/working/g1_v3"
LOG = f"{SAIDA}/{RUN}.log"

assert re.fullmatch(r"[A-Za-z0-9._-]+", RUN), "RUN aceita só letras, números, '.', '-' e '_'"
assert RETOMA in (0, 1), "RETOMA é 0 ou 1"
assert re.fullmatch(r"model_\d+\.pt", CKPT), "CKPT tem a forma model_<iteração>.pt"
print(f"RUN={RUN}  RETOMA={RETOMA}  ITERS={ITERS}  NUM_ENVS={NUM_ENVS}  TIMEOUT_H={TIMEOUT_H}")

In [ ]:
# Código: clone novo da branch g1_v3. Apaga só o clone; logs e checkpoints ficam em SAIDA.
import shutil, subprocess

shutil.rmtree(RAIZ, ignore_errors=True)
r = subprocess.run(["git", "clone", "--branch", "g1_v3", "--depth", "1",
                    "https://github.com/JoaoBornelli/g1_training.git", RAIZ], capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:])
assert r.returncode == 0, "clone falhou: confira Settings → Internet = On"
print(subprocess.run(["git", "-C", RAIZ, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Instalação: mjlab e matplotlib. Depois confere a GPU com uma conta de verdade na CUDA.
import subprocess, sys
import importlib.metadata as md

print(sys.version)
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-warn-conflicts",
                    "mjlab==1.5.3", "matplotlib"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1200:])
assert r.returncode == 0, "pip falhou (veja a saída acima)"

# Em subprocesso: o pip pode ter trocado o torch. A soma na GPU pega a placa que o torch não suporta (P100).
chk = subprocess.run([sys.executable, "-c",
                      "import torch; assert torch.cuda.is_available(); "
                      "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), "
                      "round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB', "
                      "torch.ones(8, device='cuda').sum().item())"],
                     capture_output=True, text=True)
print("CUDA (torch):", chk.stdout.strip() or chk.stderr[-600:])
assert chk.returncode == 0, "torch sem GPU utilizável: use Settings → Accelerator = GPU T4 x2"

for pacote in ["mjlab", "rsl-rl-lib", "torch", "mujoco", "warp-lang", "matplotlib"]:
    print(f"{pacote}: {md.version(pacote)}")

In [ ]:
# Checkpoint da retomada: copia de /kaggle/input para a pasta que o entrada.sh aceita
# (AAAA-MM-DD_HH-MM-SS_<RUN> com model_*.pt). Se a corrida já tem checkpoint em SAIDA, não copia nada.
import glob, hashlib, os, re, shutil

EXP = f"{SAIDA}/g1_v3"
padrao = r"\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_" + re.escape(RUN)
com_ckpt = sorted(p for p in glob.glob(f"{EXP}/*_{RUN}")
                  if re.fullmatch(padrao, os.path.basename(p)) and glob.glob(f"{p}/model_*.pt"))
if RETOMA == 0:
    print("corrida nova: nenhum checkpoint é usado")
elif com_ckpt:
    print("a retomada usa a pasta que já existe:", com_ckpt[-1])
    print(sorted(os.path.basename(m) for m in glob.glob(f"{com_ckpt[-1]}/model_*.pt"))[-3:])
else:
    achados = sorted(a for a in glob.glob(f"/kaggle/input/**/{CKPT}*", recursive=True) if os.path.isfile(a))
    assert len(achados) == 1, f"esperava 1 arquivo {CKPT}* em /kaggle/input, achei {achados}: anexe o Dataset em Add Input"
    md5 = hashlib.md5(open(achados[0], "rb").read()).hexdigest()
    print(achados[0], "| md5", md5)
    assert CKPT_MD5 in ("", md5), "md5 diferente do arquivo local: o upload do Dataset mudou o checkpoint"
    destino = f"{EXP}/2026-11-18_00-00-00_{RUN}"
    os.makedirs(destino, exist_ok=True)
    shutil.copy(achados[0], f"{destino}/{CKPT}")
    print("checkpoint em", f"{destino}/{CKPT}")

In [ ]:
# Lança o treino pelo entrada.sh, solto do kernel. Sem vídeo e sem TensorBoard.
import os, subprocess

vivo = subprocess.run(["pgrep", "-f", "g1_v3.train"], capture_output=True, text=True).stdout.split()
if vivo:
    print("JÁ EXISTE treino rodando (PID", " ".join(vivo), "). NÃO lancei outro.")
else:
    env = os.environ | {"RUN": RUN, "RETOMA": str(RETOMA), "ITERS": str(ITERS), "NUM_ENVS": str(NUM_ENVS),
                        "VIDEO": "0", "TENSORBOARD": "0", "SAIDA": SAIDA}

    # Validação sem GPU: mostra o comando e pega erro de entrada antes de lançar.
    seco = subprocess.run(["bash", f"{RAIZ}/g1_v3/container/entrada.sh"],
                          env=env | {"SECO": "1"}, capture_output=True, text=True)
    print(seco.stdout, seco.stderr)
    assert seco.returncode == 0, "validação SECO falhou: nada foi lançado"

    os.makedirs(SAIDA, exist_ok=True)
    log = open(LOG, "ab")
    proc = subprocess.Popen(["bash", "-c", f"timeout {TIMEOUT_H}h bash {RAIZ}/g1_v3/container/entrada.sh"],
                            env=env | {"SECO": "0"}, stdout=log, stderr=subprocess.STDOUT,
                            start_new_session=True, cwd=RAIZ)
    print("treino lançado, PID", proc.pid)
    print("log:", LOG)

In [ ]:
# Acompanha o log até o treino acabar. Esta célula segura a sessão do Save & Run All enquanto o treino roda.
# Interromper a célula NÃO para o treino.
import subprocess, time
from IPython.display import clear_output

try:
    while True:
        if "proc" in globals():
            vivo = proc.poll() is None
        else:
            vivo = subprocess.run(["pgrep", "-f", "g1_v3.train"], capture_output=True).returncode == 0
        clear_output(wait=True)
        print(time.strftime("%H:%M:%S"), "| processo do treino vivo:", "sim" if vivo else "NÃO")
        print(subprocess.run(["tail", "-n", "45", LOG], capture_output=True, text=True).stdout)
        if not vivo:
            break
        time.sleep(60)
    print("checkpoints:", subprocess.run(f"ls {SAIDA}/g1_v3/*_{RUN}/model_*.pt | sort -V | tail -3",
                                         shell=True, capture_output=True, text=True).stdout)
except KeyboardInterrupt:
    pass